In [10]:
import random
import sys

def score(motifs):

    if not motifs:
        return 0

    k = len(motifs[0]) # Length of k-mer
    t = len(motifs)    # Number of DNA strings (motifs)
    consensus = ""

    # Step 1: Determine the consensus string

    for j in range(k):
        counts = {'A': 0, 'C': 0, 'G': 0, 'T': 0}
        for i in range(t):
            counts[motifs[i][j]] += 1

        consensus += max(counts, key=counts.get)

    # Step 2: Calculate the total score (number of mismatches)
    total_score = 0
    for i in range(t):
        for j in range(k):
            if motifs[i][j] != consensus[j]:
                total_score += 1
    return total_score

def profile_with_pseudocounts(motifs):
    """
    Computes a profile matrix from a set of motifs, using pseudocounts.
    Pseudocounts (adding 1 to each nucleotide count) prevent zero probabilities.
    """
    if not motifs:
        return {}

    k = len(motifs[0])
    t = len(motifs)


    profile = {'A': [1.0]*k, 'C': [1.0]*k, 'G': [1.0]*k, 'T': [1.0]*k}


    for i in range(t):
        for j in range(k):
            nucleotide = motifs[i][j]
            profile[nucleotide][j] += 1.0


    for nucleotide_type in profile:
        for j in range(k):
            profile[nucleotide_type][j] /= float(t + 4)

    return profile

def get_probability(kmer, profile):

    probability = 1.0
    for i, nucleotide in enumerate(kmer):

        probability *= profile[nucleotide][i]
    return probability

def motifs_from_profile(profile, dna, k):

    best_kmers = []
    for dna_string in dna:
        max_prob = -1.0
        best_kmer_in_string = ""


        for i in range(len(dna_string) - k + 1):
            kmer = dna_string[i:i+k]
            prob = get_probability(kmer, profile)


            if prob > max_prob:
                max_prob = prob
                best_kmer_in_string = kmer

        best_kmers.append(best_kmer_in_string)
    return best_kmers

def randomized_motif_search_single_run(dna, k, t):

    # Step 1: Randomly select k-mers Motifs
    motifs = []
    for s in dna:

        start_index = random.randint(0, len(s) - k)
        motifs.append(s[start_index : start_index + k])


    best_motifs = list(motifs)


    while True:

        profile = profile_with_pseudocounts(motifs)


        motifs = motifs_from_profile(profile, dna, k)


        if score(motifs) < score(best_motifs):

            best_motifs = list(motifs)
        else:

            return best_motifs

def run_randomized_motif_search_multiple_times(dna, k, t, num_runs=1000):

    overall_best_motifs = []
    overall_best_score = float('inf')

    for run_count in range(num_runs):

        current_best_motifs = randomized_motif_search_single_run(dna, k, t)
        current_score = score(current_best_motifs)


        if current_score < overall_best_score:
            overall_best_score = current_score
            overall_best_motifs = current_best_motifs


    return overall_best_motifs

if __name__ == "__main__":

    input_filename = "/content/rosalind_ba2f (3).txt"

    try:
        with open(input_filename, 'r') as f:

            k, t = map(int, f.readline().split())


            dna_strings = [line.strip() for line in f]


        result_motifs = run_randomized_motif_search_multiple_times(dna_strings, k, t, num_runs=1000)


        for motif in result_motifs:
            print(motif)

    except FileNotFoundError:
        print(f"Error: The file '{input_filename}' was not found. Please ensure the path is correct and the file exists.")
    except Exception as e:
        print(f"An error occurred: {e}")

TGTATAATCGGTTTG
TCAGAAGTCCGTTTG
TCAATGGTCCGAGGG
TCAATGTGACGTTTG
ATTATGGTCCGTTTG
TTTGTGGTCCGTTTG
TCAATATGCCGTTTG
TCAAACCTCCGTTTG
TCAAGAATCCGTTTG
TCAATGGTCGCCTTG
TCAATCAACCGTTTG
CTAATGGTCCGTTTA
TCAATGGTCCGTAAA
TCAATGGTCCACCTG
TCAATGGTTAATTTG
TCAATGGCATGTTTG
CCAATGGTACGTGTG
TCTGAGGTCCGTTTG
ACAATGGTCCGTTAT
TCAATGAGACGTTTG
